# LangGraph Module · Day 06 — Multi-Step Planner (Plan-and-Execute)

**Phase 1 · Module 1: LangGraph & LangChain Deep Dive** · ~2.5 hrs

### Today's tasks (from the plan)
- [ ] Build a **Plan-and-Execute** agent using LangGraph
- [ ] Implement **planner node + executor node + replanning**
- [ ] Test with a **multi-document summarisation** task

🐍 **Python (30 min):** *Async Context Managers* — wrap the LLM client lifecycle so one session serves every node.

> Builds on Day 04 (tools) and Day 05 (HITL). Ref: LangGraph tutorial *'Plan-and-execute'*; Harrison Chase, re:Invent 2024.

## 0. Setup

Plan-and-Execute is a **graph structure**, not a model feature. To *see the structure clearly* this notebook uses **deterministic mock** planner/executor functions first — so it runs with **no API key**. The final section swaps in a real Gemini model, guarded by `HAS_KEY`.

Install (if needed): `pip install langgraph langchain-google-genai python-dotenv`

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
HAS_KEY = bool(os.getenv('GOOGLE_API_KEY'))
print('langgraph plan-and-execute notebook')
print('GOOGLE_API_KEY:', 'loaded ✅' if HAS_KEY else 'MISSING ❌ (only the real-LLM section is skipped)')

langgraph plan-and-execute notebook
GOOGLE_API_KEY: loaded ✅


## 1. The one big idea — *think first, then do*

A **ReAct** agent (Day 04) decides its next action **one step at a time**: call a tool, look at the result, think, call another. Great for short tasks — but on a long job it drifts, forgets the overall goal, and burns tokens re-reasoning from scratch every step.

**Plan-and-Execute** splits the brain in two:

| | ReAct (Day 04) | Plan-and-Execute (today) |
|--|----------------|--------------------------|
| Decides | one step at a time | a **whole plan** up front |
| Keeps the goal | in each prompt, re-reasoned | in an explicit **plan list** |
| Long tasks | drifts / loops | stays on track |
| Adapts to surprises | naturally | via an explicit **replan** step |

**Analogy:** ReAct is wandering a city turn-by-turn asking *"where next?"*. Plan-and-Execute writes the itinerary first, does step 1, then checks the map: *"still on track, or re-route?"* That check is **replanning** — the heart of today's lesson.

The loop we're building:

```mermaid
flowchart LR
    START([START]) --> P[planner]
    P --> E[executor]
    E --> R{replan}
    R -->|steps left| E
    R -->|done| DONE([END])
```

## 2. The state — what flows between the nodes

Every node reads and updates this shared `TypedDict` (Day 02). Four fields carry the whole pattern:

- **`input`** — the original goal (never changes).
- **`plan`** — the list of steps *still to do*.
- **`past_steps`** — an append-only log of `(step, result)`. Uses the `operator.add` **reducer** so each node's contribution is *appended*, not overwritten.
- **`response`** — the final answer, set only when we're done.

In [2]:
from typing import TypedDict, Annotated, List, Tuple
import operator

class PlanExecute(TypedDict):
    input: str                                          # the goal (fixed)
    plan: List[str]                                     # steps left to do
    past_steps: Annotated[List[Tuple[str, str]], operator.add]   # append-only log
    response: str                                       # final answer

print('state schema ready:', list(PlanExecute.__annotations__))

state schema ready: ['input', 'plan', 'past_steps', 'response']


☝️ `past_steps` is `Annotated[..., operator.add]` — the same list-append reducer from Day 02. Without it, each executor step would **overwrite** the log; with it, results **accumulate**. That accumulated log is what the replanner reads to decide what's left.

## 3. The task — summarise multiple documents

A realistic banking job: **summarise three policy documents, then produce one combined brief.** This is a natural multi-step task — you can't do it in a single tool call, and the plan has a clear order (summarise each, *then* combine).

In [3]:
DOCS = {
    'credit_policy':  'Credit limits above £50,000 require risk-officer sign-off. '
                      'Reviews are quarterly. Defaults over 90 days trigger escalation.',
    'fraud_policy':   'Transactions over £10,000 to new payees are held for review. '
                      'Card-not-present fraud is reimbursed within 5 working days.',
    'aml_policy':     'Cash deposits over £8,000 require source-of-funds checks. '
                      'Suspicious activity is reported to the FCA within 24 hours.',
}
print('documents to summarise:', list(DOCS))

documents to summarise: ['credit_policy', 'fraud_policy', 'aml_policy']


## 4. The **planner** node — turn a goal into an ordered list

The planner's only job: read the goal and emit a **numbered list of concrete steps**. Here it's deterministic (one step per doc, plus a combine step) so you can see the structure; §10 shows a real LLM writing the plan.

In [4]:
def planner(state: PlanExecute) -> dict:
    goal = state['input']
    # A real planner asks an LLM for this list. Deterministic here for clarity.
    steps = [f'Summarise the {name} document' for name in DOCS]
    steps.append('Combine the summaries into one compliance brief')
    print('  [planner] built a', len(steps), 'step plan')
    for i, s in enumerate(steps, 1):
        print(f'            {i}. {s}')
    return {'plan': steps}

demo = planner({'input': 'Summarise our policy docs', 'plan': [], 'past_steps': [], 'response': ''})

  [planner] built a 4 step plan
            1. Summarise the credit_policy document
            2. Summarise the fraud_policy document
            3. Summarise the aml_policy document
            4. Combine the summaries into one compliance brief


☝️ The planner writes the **entire** plan once. From here the agent just works the list — it doesn't re-plan from scratch every step (that's ReAct's expensive habit). It only revisits the plan at the **replan** step, and only if needed.

## 5. The **executor** node — do exactly **one** step

The executor takes the **first** step off the plan, does it, and records `(step, result)` in `past_steps`. One step per graph tick — that's what lets us re-check progress between steps.

In [5]:
def _summarise(name: str) -> str:
    text = DOCS[name]
    first_rule = text.split('.')[0].strip()          # crude 'summary' = first rule
    return f'{name}: {first_rule}.'

def executor(state: PlanExecute) -> dict:
    step = state['plan'][0]                            # the next step to do
    if step.startswith('Combine'):
        summaries = [result for _, result in state['past_steps']]
        result = 'COMBINED BRIEF -> ' + ' | '.join(summaries)
    else:
        name = step.replace('Summarise the ', '').replace(' document', '')
        result = _summarise(name)
    print(f'  [executor] did: {step!r}')
    # append this (step, result) to the log via the operator.add reducer
    return {'past_steps': [(step, result)]}

☝️ The executor returns only `{'past_steps': [(step, result)]}`. Because of the `operator.add` reducer, LangGraph **appends** that one tuple to the running log instead of replacing it. Notice it does **not** touch `plan` — removing the finished step is the replanner's job, next.

## 6. The **replan** node — the heart of the pattern

After each step the replanner asks: *"given what I've done, what's **left**?"* It has three possible verdicts:

1. **More steps remain** → drop the finished step, keep going.
2. **All steps done** → write the final `response`.
3. *(Real agents)* **the world changed** → rewrite the remaining plan (add/remove/reorder steps).

That third power — rewriting the plan mid-flight — is why it's called *re*-planning, and it's what makes the agent robust to surprises a fixed plan can't handle.

In [6]:
def replan(state: PlanExecute) -> dict:
    remaining = state['plan'][1:]                      # drop the step just executed
    if not remaining:
        # nothing left -> the last result IS the final answer
        final = state['past_steps'][-1][1]
        print('  [replan] plan complete -> writing final response')
        return {'response': final, 'plan': []}
    print(f'  [replan] {len(remaining)} step(s) left ->', remaining[0])
    return {'plan': remaining}

☝️ On the last step, the replanner sets `response` (which ends the loop). Otherwise it hands back a **shorter** `plan` and the loop continues. A real replanner would ask an LLM *"here's the goal, here's what's done — give me the updated remaining steps"*, letting it insert or drop steps. Same shape, smarter contents (see §10).

## 7. The exit condition — a conditional edge

How does the loop know to stop? A **conditional edge** (Day 02) reads state after `replan`: if `response` is set → `END`; otherwise → back to `executor` for the next step.

In [7]:
from langgraph.graph import StateGraph, START, END

def should_end(state: PlanExecute) -> str:
    return END if state['response'] else 'executor'

print('router ready: response set -> END, else -> executor')

router ready: response set -> END, else -> executor


☝️ This one function is the whole loop control: `replan` decides *whether* we're done by setting `response`, and `should_end` **routes** on that decision. Clean separation — the node computes, the edge routes.

## 8. Assemble the graph

Wire the three nodes into the loop from §1:
`START → planner → executor → replan → (executor | END)`.

In [8]:
builder = StateGraph(PlanExecute)
builder.add_node('planner', planner)
builder.add_node('executor', executor)
builder.add_node('replan', replan)

builder.add_edge(START, 'planner')
builder.add_edge('planner', 'executor')
builder.add_edge('executor', 'replan')
builder.add_conditional_edges('replan', should_end, ['executor', END])

graph = builder.compile()
print('compiled. nodes:', list(graph.get_graph().nodes))

compiled. nodes: ['__start__', 'planner', 'executor', 'replan', '__end__']


In [9]:
# LangGraph emits the diagram for you as Mermaid source - no extra deps.
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	executor(executor)
	replan(replan)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	executor --> replan;
	planner --> executor;
	replan -.-> __end__;
	replan -.-> executor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



That Mermaid source renders as the loop below (planner runs once; executor↔replan cycle until `END`):

```mermaid
flowchart TD
    START([__start__]) --> planner
    planner --> executor
    executor --> replan
    replan -.->|response empty| executor
    replan -.->|response set| END([__end__])
```

☝️ `draw_mermaid()` is generated **from the compiled graph itself** — so the picture can never drift from the real wiring. Paste it into any Mermaid renderer (GitHub, this notebook, mermaid.live) to see the loop.

## 9. Run it — watch plan → execute → replan loop

`stream` lets us watch each node fire in order. Watch the plan shrink by one on every `replan` until `response` is set and the loop exits.

In [10]:
init = {'input': 'Summarise our credit, fraud and AML policies into one brief',
        'plan': [], 'past_steps': [], 'response': ''}

final_state = None
for event in graph.stream(init, stream_mode='values'):
    final_state = event

print('\n=== FINAL RESPONSE ===')
print(final_state['response'])
print('\nsteps executed:', len(final_state['past_steps']))

  [planner] built a 4 step plan
            1. Summarise the credit_policy document
            2. Summarise the fraud_policy document
            3. Summarise the aml_policy document
            4. Combine the summaries into one compliance brief
  [executor] did: 'Summarise the credit_policy document'
  [replan] 3 step(s) left -> Summarise the fraud_policy document
  [executor] did: 'Summarise the fraud_policy document'
  [replan] 2 step(s) left -> Summarise the aml_policy document
  [executor] did: 'Summarise the aml_policy document'
  [replan] 1 step(s) left -> Combine the summaries into one compliance brief
  [executor] did: 'Combine the summaries into one compliance brief'
  [replan] plan complete -> writing final response

=== FINAL RESPONSE ===
COMBINED BRIEF -> credit_policy: Credit limits above £50,000 require risk-officer sign-off. | fraud_policy: Transactions over £10,000 to new payees are held for review. | aml_policy: Cash deposits over £8,000 require source-of-funds check

☝️ Four steps ran — three summaries **then** the combine — each followed by a replan check, exactly as planned. The `past_steps` log carried every intermediate result forward so the final `Combine` step had them all. **That accumulation is why plan-execute beats step-by-step ReAct on multi-part tasks.**

## 10. Real LLM — an actual planner + replanner (needs `GOOGLE_API_KEY`)

Now swap the mock brains for Gemini. The graph shape is **identical** — only the `planner` and `replan` node bodies change to call a model with **structured output** (so we get a clean `List[str]` plan back, not free text).

In [11]:
if HAS_KEY:
    from langchain_google_genai import ChatGoogleGenerativeAI
    from pydantic import BaseModel, Field

    model = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)

    class Plan(BaseModel):
        """An ordered list of steps to accomplish the goal."""
        steps: List[str] = Field(description='steps in order, each a single action')

    planner_llm = model.with_structured_output(Plan)

    corpus = '\n'.join(f'- {k}: {v}' for k, v in DOCS.items())

    def llm_planner(state: PlanExecute) -> dict:
        msg = (f'Goal: {state["input"]}\n\nAvailable documents:\n{corpus}\n\n'
               'Produce a short ordered plan: one step per document, then a final combine step.')
        plan = planner_llm.invoke(msg).steps
        print('  [llm planner]', len(plan), 'steps')
        return {'plan': plan}

    def llm_executor(state: PlanExecute) -> dict:
        step = state['plan'][0]
        done = '\n'.join(f'{s} -> {r}' for s, r in state['past_steps']) or '(none yet)'
        msg = (f'Documents:\n{corpus}\n\nAlready done:\n{done}\n\n'
               f'Now do ONLY this step in one sentence: {step}')
        result = model.invoke(msg).content
        print(f'  [llm executor] {step[:40]}...')
        return {'past_steps': [(step, result)]}

    def llm_replan(state: PlanExecute) -> dict:
        remaining = state['plan'][1:]
        if not remaining:
            return {'response': state['past_steps'][-1][1], 'plan': []}
        return {'plan': remaining}

    b = StateGraph(PlanExecute)
    b.add_node('planner', llm_planner)
    b.add_node('executor', llm_executor)
    b.add_node('replan', llm_replan)
    b.add_edge(START, 'planner')
    b.add_edge('planner', 'executor')
    b.add_edge('executor', 'replan')
    b.add_conditional_edges('replan', should_end, ['executor', END])
    llm_graph = b.compile()

    out = llm_graph.invoke({'input': 'Summarise our credit, fraud and AML policies into one brief',
                            'plan': [], 'past_steps': [], 'response': ''})
    print('\n=== LLM FINAL BRIEF ===\n', out['response'])
else:
    print('skipped (no API key) - the mock graph in §9 is the real, runnable deliverable')

  [llm planner] 4 steps
  [llm executor] Summarise the credit policy, including r...
  [llm executor] Summarise the fraud policy, covering new...
  [llm executor] Summarise the AML policy, detailing cash...
  [llm executor] Combine the summaries of the credit, fra...

=== LLM FINAL BRIEF ===
 The credit policy mandates risk-officer sign-off for limits over £50,000, quarterly reviews, and 90-day default escalation; the fraud policy holds new payee transactions over £10,000 and reimburses card-not-present fraud within 5 days; and the AML policy requires source-of-funds checks for cash deposits over £8,000 and 24-hour FCA reporting for suspicious activity.


☝️ Same three nodes, same loop, same `should_end` router — the **structure is the reusable asset**. The mock and the LLM version are interchangeable because plan-execute is about *graph shape*, not about the model. (And per today's Python notebook, in production you'd open the model client once inside an `async with` so every node reuses one session.)

## 11. Recap

| Concept | What it is | Why it's used |
|---------|-----------|---------------|
| **Plan-and-Execute** | plan the whole task, then work the list | stays on-goal over long tasks where ReAct drifts |
| **planner node** | goal → ordered list of steps | commit to a strategy once, not per-step |
| **executor node** | do exactly one step, log the result | small units → progress is checkable between steps |
| **`past_steps` + `operator.add`** | append-only result log | intermediate results accumulate for later steps |
| **replan node** | decide what's left / rewrite the plan | adapt to reality; set `response` to finish |
| **conditional edge (`should_end`)** | route on whether `response` is set | closes the execute↔replan loop or exits |
| **structured output (`Plan`)** | force the LLM to return `List[str]` | a clean plan the graph can iterate, not prose |

**The pattern in one line:** *plan once, execute one step, re-check, repeat until done.*